<a href="https://colab.research.google.com/github/Innotech-Manipal-TP26/Ananyaraj2ndYear/blob/main/Week4/ResidualCNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split
import time

torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [2]:
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

train_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform_train)

test_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform_test)

100%|██████████| 170M/170M [00:37<00:00, 4.54MB/s]


In [3]:
train_size = int(0.9 * len(train_dataset))
val_size = len(train_dataset) - train_size

train_dataset, val_dataset = random_split(train_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

In [4]:
class ResidualBlock(nn.Module):

    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels, out_channels, 3,
                               stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)

        self.conv2 = nn.Conv2d(out_channels, out_channels, 3,
                               padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.relu = nn.ReLU()

        self.shortcut = nn.Sequential()

        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels,out_channels,1,stride=stride,bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self,x):

        identity = self.shortcut(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        out += identity
        out = self.relu(out)

        return out

In [5]:
class ResidualCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(3,64,3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU()
        )

        self.layer1 = ResidualBlock(64,64)
        self.layer2 = ResidualBlock(64,128,stride=2)
        self.layer3 = ResidualBlock(128,256,stride=2)

        self.pool = nn.AdaptiveAvgPool2d((1,1))
        self.fc = nn.Linear(256,10)

    def forward(self,x):

        x = self.stem(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)

        x = self.pool(x)

        x = torch.flatten(x,1)
        x = self.fc(x)

        return x

In [9]:
def train_model(model, train_loader, val_loader, epochs=20):

    model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)

    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=0.003,
        steps_per_epoch=len(train_loader),
        epochs=epochs
    )

    # 🔥 ADD THESE
    train_losses = []
    val_losses = []
    train_accs = []
    val_accs = []

    for epoch in range(epochs):

        model.train()

        train_loss = 0
        correct = 0
        total = 0

        for images, labels in train_loader:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()
            scheduler.step()

            train_loss += loss.item()

            _, pred = outputs.max(1)
            correct += pred.eq(labels).sum().item()
            total += labels.size(0)

        train_acc = correct / total
        train_loss = train_loss / len(train_loader)

        val_acc, val_loss = evaluate(model, val_loader)

        gap = train_acc - val_acc

        # 🔥 STORE VALUES
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        train_accs.append(train_acc)
        val_accs.append(val_acc)

        print(f"Epoch {epoch+1} | Train Acc {train_acc:.4f} | Val Acc {val_acc:.4f} | Gap {gap:.4f}")

    # 🔥 IMPORTANT RETURN
    return train_losses, val_losses, train_accs, val_accs

In [7]:
def evaluate(model, loader):

    model.eval()

    criterion = nn.CrossEntropyLoss()

    loss = 0
    correct = 0
    total = 0

    with torch.no_grad():

        for images,labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            l = criterion(outputs,labels)
            loss += l.item()

            _,pred = outputs.max(1)

            correct += pred.eq(labels).sum().item()
            total += labels.size(0)

    acc = correct/total

    return acc, loss

In [10]:
resnet_model = ResidualCNN()
train_losses, val_losses, train_accs, val_accs = train_model(
    resnet_model,
    train_loader,
    val_loader,
    epochs=20
)

Epoch 1 | Train Acc 0.4271 | Val Acc 0.4712 | Gap -0.0441
Epoch 2 | Train Acc 0.5746 | Val Acc 0.5160 | Gap 0.0586
Epoch 3 | Train Acc 0.6427 | Val Acc 0.5962 | Gap 0.0465
Epoch 4 | Train Acc 0.6866 | Val Acc 0.6760 | Gap 0.0106
Epoch 5 | Train Acc 0.7326 | Val Acc 0.7040 | Gap 0.0286
Epoch 6 | Train Acc 0.7686 | Val Acc 0.7432 | Gap 0.0254
Epoch 7 | Train Acc 0.7956 | Val Acc 0.7764 | Gap 0.0192
Epoch 8 | Train Acc 0.8197 | Val Acc 0.7542 | Gap 0.0655
Epoch 9 | Train Acc 0.8356 | Val Acc 0.8018 | Gap 0.0338
Epoch 10 | Train Acc 0.8480 | Val Acc 0.8182 | Gap 0.0298
Epoch 11 | Train Acc 0.8652 | Val Acc 0.8284 | Gap 0.0368
Epoch 12 | Train Acc 0.8775 | Val Acc 0.8274 | Gap 0.0501
Epoch 13 | Train Acc 0.8892 | Val Acc 0.8482 | Gap 0.0410
Epoch 14 | Train Acc 0.9015 | Val Acc 0.8732 | Gap 0.0283
Epoch 15 | Train Acc 0.9125 | Val Acc 0.8758 | Gap 0.0367
Epoch 16 | Train Acc 0.9256 | Val Acc 0.8908 | Gap 0.0348
Epoch 17 | Train Acc 0.9359 | Val Acc 0.8948 | Gap 0.0411
Epoch 18 | Train Acc 0